# Контроль качества данных эксперимента 3

**Статус:** техническая инвентаризация и диагностический ноутбук
реокардиомонитора РНЦХ. Функции сердца здесь не рассчитываются.

Паспорт серии приведён в
[`10.10`](10.10_Паспорт_эксперимента_3.md). Дата остаётся открытым полем до
сверки с первичным протоколом. Постороннее исследование на другом приборе не
включается: ноутбук обрабатывает только записи, явно перечисленные во внешней
конфигурации.

Численные наблюдения перепроверены 27.08.2026 по четырём основным CSV из
локальной конфигурации и их полным контрольным суммам. Отслеживаемые outputs
очищены. Обезличенная подробная сводка хранится во внешнем `derived_root` в
`exp03/qc/10.11_primary_signal_diagnostic.candidate.json` со статусом
`pending_manual_review`.

Ориентировочные дыхательные интервалы из старых графиков не используются как
количественная разметка. Последовательность режимов каждой записи должна быть
подтверждена первичным протоколом.


In [ ]:
# Импорты и внешний контракт данных
import hashlib
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

CONFIG_ENV = "KALMYKOV_EXP03_CONFIG"
config_value = os.environ.get(CONFIG_ENV)
if not config_value:
    raise RuntimeError(
        f"Задайте {CONFIG_ENV}: путь к локальному JSON по схеме "
        "config/exp03_paths.example.json"
    )

CONFIG_PATH = Path(config_value).expanduser().resolve()
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
DATA_ROOT = Path(CONFIG["data_root"]).expanduser().resolve()
CSV_ROOT = (DATA_ROOT / CONFIG["csv_subdir"]).resolve()
CSV_ROOT.relative_to(DATA_ROOT)

RECORDINGS = CONFIG["recordings"]
record_ids = [item["record_id"] for item in RECORDINGS]
relative_paths = [item["relative_path"] for item in RECORDINGS]
if not RECORDINGS or len(record_ids) != len(set(record_ids)):
    raise ValueError("recordings должен содержать уникальные непустые record_id")
if len(relative_paths) != len(set(relative_paths)):
    raise ValueError("Один relative_path нельзя назначать нескольким record_id")

SOURCE_COLUMNS = CONFIG["source_columns"]
CANONICAL_COLUMNS = [
    "time_s", "rheo_1_mohm", "base_1_ohm", "qs_1_ohm",
    "ecg_v", "rheo_2_mohm", "base_2_ohm", "qs_2_ohm",
]
if len(SOURCE_COLUMNS) != len(CANONICAL_COLUMNS):
    raise ValueError("source_columns должен описывать ровно восемь столбцов CSV")

ACTIVE_THRESHOLD_OHM = float(CONFIG["active_channel_threshold_ohm"])
if not np.isfinite(ACTIVE_THRESHOLD_OHM) or ACTIVE_THRESHOLD_OHM <= 0:
    raise ValueError("active_channel_threshold_ohm должен быть положительным")

plt.rcParams.update({
    "figure.figsize": (15, 9),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
    "axes.titlesize": 13,
})


def resolve_record_path(relative_path):
    path = (DATA_ROOT / relative_path).resolve()
    path.relative_to(CSV_ROOT)
    if not path.is_file():
        raise FileNotFoundError(f"Нет файла для записи: {relative_path}")
    return path


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def read_record(path):
    frame = pd.read_csv(path)
    if list(frame.columns) != SOURCE_COLUMNS:
        raise ValueError(
            "Схема CSV не совпадает с source_columns внешней конфигурации"
        )
    frame.columns = CANONICAL_COLUMNS
    frame = frame.apply(pd.to_numeric, errors="raise")
    values = frame.to_numpy(dtype=float)
    if len(frame) < 2 or not np.isfinite(values).all():
        raise ValueError("CSV пуст, слишком короток или содержит нечисловые значения")
    dt = np.diff(frame["time_s"].to_numpy(dtype=float))
    if not np.all(dt > 0):
        raise ValueError("TIME должен строго возрастать")
    return frame


## 1. Инвентаризация и соответствие файлов

В анализ входят только обезличенные `record_id`, явно перечисленные в
`KALMYKOV_EXP03_CONFIG`. Относительный путь обязан находиться внутри
разрешённого `csv_subdir`. Остальные CSV учитываются только числом при проверке
полноты конфигурации и не анализируются.

Полный SHA-256 определяет байт-в-байт совпадающие записи. Канонический файл
задаётся конфигурацией, а не выбирается автоматически по длине пути.


In [ ]:
# Инвентаризация разрешённой области и явно включённых записей
discovered = sorted(CSV_ROOT.rglob("*.csv"))
discovered_hashes = {path: file_sha256(path) for path in discovered}
copy_count_by_hash = pd.Series(list(discovered_hashes.values())).value_counts()

inventory_rows = []
listed_paths = set()
for spec in RECORDINGS:
    path = resolve_record_path(spec["relative_path"])
    listed_paths.add(path)
    data = read_record(path)
    dt = np.diff(data["time_s"].to_numpy(dtype=float))
    sha256 = discovered_hashes.get(path, file_sha256(path))
    inventory_rows.append({
        "record_id": spec["record_id"],
        "path": path,
        "sha256": sha256,
        "sha256_prefix": sha256[:16],
        "copies_in_allowed_root": int(copy_count_by_hash.get(sha256, 1)),
        "rows": len(data),
        "duration_s": float(data["time_s"].iloc[-1] - data["time_s"].iloc[0]),
        "fs_hz": float(1.0 / np.median(dt)),
        "role": spec.get("role", "не указана"),
        "protocol_reference": spec.get("protocol_reference", "не указана"),
        "expected_active_channels": spec.get("expected_active_channels", []),
    })

inventory = pd.DataFrame(inventory_rows)
inventory["duplicate_in_list"] = inventory.duplicated("sha256", keep=False)
if inventory["duplicate_in_list"].any():
    duplicate_ids = inventory.loc[inventory["duplicate_in_list"], "record_id"].tolist()
    raise ValueError(f"В recordings перечислены дублирующие записи: {duplicate_ids}")

unlisted_count = len(set(discovered) - listed_paths)
mapping_table = inventory[[
    "record_id", "role", "protocol_reference", "expected_active_channels",
    "duration_s", "fs_hz", "copies_in_allowed_root", "sha256_prefix",
]].copy()
mapping_table["duration_s"] = mapping_table["duration_s"].round(2)
mapping_table["fs_hz"] = mapping_table["fs_hz"].round(2)
mapping_table.columns = [
    "Запись", "Роль", "Ссылка на протокол", "Ожидаемые активные каналы",
    "Длительность, с", "Частота, Гц", "Копий в разрешённой области",
    "SHA-256, начало",
]
display(mapping_table.style.hide(axis="index").set_properties(**{"text-align": "left"}))
print(f"Явно включено записей: {len(inventory)}")
print(f"Других CSV в разрешённой области: {unlisted_count}")
print("Неуказанные CSV не анализируются.")


### Результат файловой проверки

- запись `14-07-01` соответствует отдельному каналу 1, ТТРКГ;
- запись `14-09-42` соответствует отдельному каналу 2, боковой сборке;
- запись `14-17-16` содержит поочерёдное отключение каналов, тогда как в
  текстовом протоколе для этой пробы указано 14:08;
- запись `14-19-52` содержит совместную дыхательную регистрацию, тогда как в
  текстовом протоколе указано 14:17;
- четыре основные записи имели байт-в-байт копии в двух каталогах разрешённой
  области данных.

Повторная проверка подтвердила соответствие назначенных ролей фактическим
состояниям каналов по техническому порогу `BASE > 5 Ом`. Установленное
несоответствие двух временных обозначений не объясняется. Соответствие
записей протокольным пробам и последовательности дыхательных команд должно
быть окончательно подтверждено по первичному журналу. Дата эксперимента по
этим временам не определяется.


In [ ]:
record_by_id = dict(zip(inventory["record_id"], inventory["path"]))
spec_by_id = {item["record_id"]: item for item in RECORDINGS}


def path_for_role(role):
    matches = [item for item in RECORDINGS if item.get("role") == role]
    if len(matches) != 1:
        raise ValueError(f"Для роли {role!r} ожидается ровно одна запись")
    return record_by_id[matches[0]["record_id"]], matches[0]["record_id"]


def channel_stats(path, channels):
    data = read_record(path)
    result = []
    for channel in channels:
        base = data[f"base_{channel}_ohm"]
        qs = data[f"qs_{channel}_ohm"]
        rheo = data[f"rheo_{channel}_mohm"]
        result.append({
            "Канал": channel,
            "BASE median, Ом": base.median(),
            "BASE p01–p99, Ом": f"{base.quantile(0.01):.2f}–{base.quantile(0.99):.2f}",
            "QS median, Ом": qs.median(),
            "QS max, Ом": qs.max(),
            "RHEO p01–p99, мОм": f"{rheo.quantile(0.01):.1f}–{rheo.quantile(0.99):.1f}",
        })
    table = pd.DataFrame(result)
    for column in ["BASE median, Ом", "QS median, Ом", "QS max, Ом"]:
        table[column] = table[column].round(2)
    return table


def plot_record(path, title, channels):
    data = read_record(path)
    time = data["time_s"]
    fs = 1.0 / np.median(np.diff(time))
    window = max(1, int(round(fs)))
    if window % 2 == 0:
        window += 1

    fig, axes = plt.subplots(
        4, 1, figsize=(15, 10), sharex=True,
        gridspec_kw={"height_ratios": [2.2, 1, 1, 1]},
    )
    colors = {1: "#2563eb", 2: "#dc2626"}
    for channel in channels:
        rheo = data[f"rheo_{channel}_mohm"]
        smooth = rheo.rolling(window, center=True, min_periods=1).median()
        axes[0].plot(time, rheo, color=colors[channel], alpha=0.16, linewidth=0.5)
        axes[0].plot(
            time, smooth, color=colors[channel], linewidth=1.7,
            label=f"канал {channel}, медиана 1 с",
        )
        axes[1].plot(
            time, data[f"base_{channel}_ohm"], color=colors[channel],
            linewidth=1.2, label=f"BASE {channel}",
        )
        axes[2].plot(
            time, data[f"qs_{channel}_ohm"], color=colors[channel],
            linewidth=1.2, label=f"QS {channel}",
        )

    axes[3].plot(time, data["ecg_v"], color="#111827", linewidth=0.7, label="ЭКГ")
    axes[0].set_ylabel("RHEO, мОм")
    axes[1].set_ylabel("BASE, Ом")
    axes[2].set_ylabel("QS, Ом")
    axes[3].set_ylabel("ЭКГ, В")
    axes[3].set_xlabel("Время от начала CSV, с")
    axes[0].set_title(title)
    for axis in axes:
        axis.legend(loc="upper right", ncol=max(1, len(channels)))
        axis.margins(x=0)
    fig.tight_layout()
    plt.show()


def active_intervals(path, threshold_ohm=ACTIVE_THRESHOLD_OHM):
    data = read_record(path)
    time = data["time_s"].to_numpy()
    base_1 = data["base_1_ohm"].to_numpy()
    base_2 = data["base_2_ohm"].to_numpy()
    state = (base_1 > threshold_ohm).astype(int) + 2 * (base_2 > threshold_ohm).astype(int)
    cuts = np.r_[0, np.flatnonzero(state[1:] != state[:-1]) + 1, len(state)]
    names = {0: "ни один", 1: "только 1", 2: "только 2", 3: "оба"}
    rows = []
    for left, right in zip(cuts[:-1], cuts[1:]):
        if time[right - 1] - time[left] < 0.25:
            continue
        rows.append({
            "Начало, с": time[left],
            "Конец, с": time[right - 1],
            "Активны по порогу": names[int(state[left])],
            "BASE1 median, Ом": np.median(base_1[left:right]),
            "BASE2 median, Ом": np.median(base_2[left:right]),
            "QS1 median, Ом": np.median(data["qs_1_ohm"].iloc[left:right]),
            "QS2 median, Ом": np.median(data["qs_2_ohm"].iloc[left:right]),
        })
    return pd.DataFrame(rows).round(2)


## 2. Отдельная запись ТТРКГ, канал 1

Назначение файла и работа только канала 1 подтверждены. Низкочастотные
изменения `BASE1` позволяют искать кандидатные спокойные участки, но точная
последовательность дыхательных режимов не подтверждена. До сверки с первичным
протоколом `mode_sequence` должна быть пустой, а серия `11.11` не должна
принимать дыхательную разметку.


In [ ]:
path, record_id = path_for_role("ttrkg_channel_1_only")
display(channel_stats(path, [1]).style.hide(axis="index"))
plot_record(path, f"{record_id}: ТТРКГ, канал 1 отдельно", [1])


**Диагностическое наблюдение.** Медиана `BASE1` равна 101,353 Ом, тогда
как в примечании протокола указано около 80 Ом. `QS1` сохраняет значение
4700 Ом на протяжении всей записи. Физический смысл этого кода и статус
предела шкалы не установлены.

В `RHEO1` повторяются уровни около −584,501 и +584,215 мОм. Доля наиболее
частых дискретных крайних уровней составляет 3,26 % отсчётов, а самый длинный
непрерывный участок — 0,57 с. Это наблюдение о записанных числах. Насыщение
измерительного тракта остаётся гипотезой; запись требует расшифровки `QS` и
проверки диапазона `RHEO`.


## 3. Отдельная запись боковой сборки, канал 2

Назначение файла и работа только канала 2 подтверждены. Низкочастотная форма
`BASE2` содержит кандидатные спокойные участки, но их физиологические метки и
порядок команд не восстановлены. Старые границы не передаются в
количественный анализ без принятой разметки `11.11`.


In [ ]:
path, record_id = path_for_role("side_channel_2_only")
display(channel_stats(path, [2]).style.hide(axis="index"))
plot_record(path, f"{record_id}: боковая сборка, канал 2 отдельно", [2])


**Диагностическое наблюдение.** Медиана `BASE2` равна 37,158 Ом и близка
к значению 36 Ом из протокола. `QS2` находится около 343 Ом и не принимает
значение 4700 Ом. `BASE1` равен нулю, что согласуется с отключением канала 1.
Близость базового уровня протоколу не заменяет калибровку прибора.

В `RHEO2` уровни около −505,679 и +505,432 мОм занимают 12,44 % отсчётов;
самый длинный непрерывный участок длится 2,31 с. До проверки измерительного
тракта эти участки нельзя использовать для оценки дыхательной или пульсовой
амплитуды. Причина повторения крайних уровней не установлена.


## 4. Запись поочерёдного отключения каналов

Кандидатные состояния определяются по порогу `BASE`, заданному во внешней
конфигурации. Порог является техническим правилом сегментации, а не
физическим критерием исправности. Переходы и подписи состояний должны быть
проверены по протоколу до использования численных различий.


In [ ]:
path, record_id = path_for_role("channel_switch_test")
switch_intervals = active_intervals(path)
display(switch_intervals.style.hide(axis="index"))
plot_record(path, f"{record_id}: поочерёдное отключение каналов", [1, 2])


**Диагностическое наблюдение.** Технические состояния по `BASE > 5 Ом`
образуют последовательность: оба канала 0,000–12,075 с; только канал 2
12,080–22,990 с; оба канала 22,995–34,025 с; только канал 1
34,030–42,805 с; оба канала 42,810–46,020 с. В последнем интервале длительностью
3,21 с оценка менее устойчива и не проходит установленный минимум 4 с.

При совместной работе медианы составляют приблизительно `BASE1=54,2 Ом` и
`BASE2=40,9 Ом`. После отключения канала 1 `BASE2` остаётся около 36,9 Ом;
после отключения канала 2 `BASE1` возрастает примерно до 92,7 Ом. Зависимость
базового уровня оставшегося канала от состояния соседнего подтверждается без
ЭКГ-разметки. Причина эффекта не определена.

Повторяющиеся крайние уровни занимают 3,45 % отсчётов `RHEO1` и 5,98 %
отсчётов `RHEO2`; непрерывные участки достигают 0,91 и 1,03 с. Поэтому
пульсовые амплитуды внутри этих интервалов пока не являются принятым
результатом. Значение `QS=4700` встречается у отключённого канала и у канала 1
при одиночной работе, однако его физическая интерпретация неизвестна.


## 5. Совместная запись обоих каналов с дыхательными манёврами

Работа обоих каналов на всей записи подтверждена. В `BASE` присутствуют
низкочастотные изменения и продолжительные спокойные участки, но точные
дыхательные режимы и их порядок не подтверждены первичным протоколом. Этот
ноутбук не назначает им физиологические метки; принятая разметка должна
поступать из `11.11`.


In [ ]:
path, record_id = path_for_role("both_channels_breathing")
display(channel_stats(path, [1, 2]).style.hide(axis="index"))
plot_record(path, f"{record_id}: оба канала, дыхательный протокол", [1, 2])


**Диагностическое наблюдение.** При совместной работе медианы равны
`BASE1=54,419 Ом` и `BASE2=40,717 Ом`; второй уровень близок к протокольному,
первый ниже указанного значения 60 Ом. `QS1` находится около 1307 Ом, а
`QS2` — около 368 Ом.

Повторяющиеся крайние уровни занимают 6,88 % отсчётов `RHEO1` и 12,58 %
отсчётов `RHEO2`; самые длинные непрерывные участки составляют 1,01 и 1,89 с.
Их временное совпадение с дыхательными изменениями не устанавливает ни
физиологический источник, ни механизм ограничения диапазона. До аппаратной
проверки и принятой дыхательной разметки эти участки исключают количественную
оценку амплитуд.


## 6. Диагностическая проверка автоматических ЭКГ-кандидатов

Для сопоставления с архивным расчётом воспроизведён детектор текущей серии
`11.12` с параметрами локальной конфигурации. Кандидатный набор характеризуют
число событий, частота по числу событий, частота по медиане RR и максимальный
интервал RR. Эти показатели не являются принятой ЭКГ-разметкой.

| Запись | Кандидаты R | Частота по числу, уд/мин | Частота по медиане RR, уд/мин | Максимальный RR, с |
|---|---:|---:|---:|---:|
| `14-07-01` | 71 | 63,2 | 76,9 | 3,62 |
| `14-09-42` | 24 | 20,8 | 49,6 | 10,46 |
| `14-17-16` | 4 | 5,2 | 6,0 | 11,96 |
| `14-19-52` | 80 | 65,2 | 75,0 | 5,66 |

Во всех наборах имеются длинные пропуски. В записи `14-17-16` четыре
автоматических события находятся не далее 0,57 с от границ технических
состояний каналов, поэтому их нельзя принимать как R-зубцы без ручной
проверки. Ни один сопроводительный файл ЭКГ не имеет статуса
`accepted`.

Архивный детектор `40.90` воспроизводит прежние количества 72, 72, 40 и 70
событий, но максимальные RR остаются равными 7,18; 5,26; 7,59 и 14,84 с.
Следовательно, старый критерий по числу событий и медианной частоте пропускал
длинные участки без надёжной детекции. Старые ансамблевые амплитуды и вывод о
сердечном происхождении не подтверждены.


## 7. Выводы и границы результата

### Подтверждённые файловые и экспериментальные факты

1. В разрешённой области данных обнаружены байт-в-байт копии четырёх основных
   записей; расчёт должен использовать каждый полный SHA-256 один раз.
2. Назначенные роли четырёх основных записей совпадают с фактическими
   состояниями каналов по техническому порогу `BASE > 5 Ом`.
3. В записи поочерёдного отключения базовый уровень оставшегося канала зависит
   от состояния другого канала. Механизм эффекта не установлен.
4. Временные обозначения двух записей не совпадают с текстовым протоколом.
5. Постороннее несинхронное исследование на другом приборе не относится к
   эксперименту 3.

### Диагностические наблюдения

- во всех активных `RHEO` основных записей имеются продолжительные повторения
  дискретных крайних уровней; насыщение является только гипотезой;
- автоматические ЭКГ-кандидаты текущего и архивного детекторов содержат
  длинные пропуски и не приняты вручную;
- отдельная запись канала 2 имеет `BASE2`, близкий к протокольному значению, и
  стабильный `QS2` без значения 4700 Ом;
- отдельная запись канала 1 имеет `BASE1` выше протокольного значения и
  постоянный код `QS1=4700 Ом`; физический смысл этого кода не установлен.

### Требования к продолжению

1. Сверить дату, временные метки и последовательность дыхательных команд по
   первичному журналу; до этого сохранять пустую `mode_sequence`.
2. Проверить `QS`, пределы диапазона, калибровку `BASE/RHEO` и межканальное
   влияние по отдельному аппаратному протоколу.
3. Выполнить ручную ЭКГ-разметку с явным контролем пропусков, ложных событий и
   участков переключения каналов.
4. Только после принятия сопроводительных файлов дыхания и ЭКГ повторить
   пульсовые ансамбли и сравнение конфигураций. Числа архивного `40.90` до
   этого не использовать.
5. Числа версии РНЦХ не использовать как поправку для версии МГТУ.


In [ ]:
# @title Файловый QC и кандидатный манифест
from record_qc import build_exp03_candidate_manifest

QC_MANIFEST_PATH, QC_MANIFEST = build_exp03_candidate_manifest(CONFIG_PATH)
QC_INCLUDED = [item for item in QC_MANIFEST["records"] if item["include"]]
QC_UNCLASSIFIED = [
    item for item in QC_MANIFEST["records"]
    if item["qc_status"] == "unclassified_not_included_pending_primary_protocol"
]
print("10.11 file_qc_status: pending_manual_review")
print("Основных включённых записей:", len(QC_INCLUDED))
print("Неклассифицированных уникальных записей:", len(QC_UNCLASSIFIED))
print("Кандидатный манифест:", QC_MANIFEST_PATH)


## Производный манифест записей

Последующие расчёты серии 40 принимают только обезличенный QC-манифест по схеме
`schemas/record_manifest.schema.json`. Он должен фиксировать точный `record_id`,
SHA-256 исходной записи, добровольца, конфигурацию, монтаж, размер боковой сборки
и **фактически**, а не ожидаемо активные каналы. До ручного принятия такого
манифеста реальный расчёт серии 40 блокируется.
